# ChaosBench to a LEAP bucket

Downloads the ChaosBench inputs (`climatology`, `lra5`, `oras5`, `era5`) from Hugging Face, extracts them, uploads them to your own Google Cloud bucket and deletes the local copy, one dataset after another.

Full guide: [`docs/DATA_SETUP.md`](../docs/DATA_SETUP.md). Run the cells **one at a time, top to bottom**. Do not use "Run All" while a transfer is running.

**Disk:** peak local use is archive + extracted data. ERA5 needs about 45 GB of free home quota (for the archive) and about 50 GB on `/tmp` (for the extracted data).

In [2]:
import os, pathlib

# --- the repo root, found from wherever this notebook runs ---
REPO = pathlib.Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
SCRIPTS = REPO / "scripts" / "chaosbench"
assert SCRIPTS.exists(), f"scripts not found at {SCRIPTS}; open this notebook from inside the PyroWatch clone"

# --- EDIT: your own bucket folder (leap-persistent, inside your username) ---
os.environ["BUCKET"] = os.environ.get("CHAOSBENCH_BUCKET", "gs://leap-persistent/cristiarsamosir/chaosbench")

# Extracted data: fast local disk (wiped if the server restarts)
os.environ["WORK"] = "/tmp/chaosbench_work"
# Downloaded archive: home storage, so it does not share the local disk with the extracted data
os.environ["PARTS_ROOT"] = os.path.expanduser("~/chaosbench_parts")
os.environ["SCRIPTS"] = str(SCRIPTS)
os.environ["LOG"] = str(REPO / "notebooks" / "transfer.log")

# Datasets, in the order they run (era5 last: it needs the most disk).
REQUIRED = "climatology lra5 oras5 era5"
# OPTION LINE: space-separated extra datasets, or "" for none.
#   control forecasts : ukmo ncep cma ecmwf
#   ensemble forecasts: ukmo_ensemble ncep_ensemble cma_ensemble ecmwf_ensemble
#   model output      : panguweather fourcastnetv2 graphcast
EXTRA_DATASETS = ""
os.environ["DATASETS"] = f"{REQUIRED} {EXTRA_DATASETS}".strip()

for k in ("BUCKET", "WORK", "PARTS_ROOT", "DATASETS"):
    print(f"{k:11}:", os.environ[k])

BUCKET     : gs://leap-persistent/cristiarsamosir/chaosbench
WORK       : /tmp/chaosbench_work
PARTS_ROOT : /home/jovyan/chaosbench_parts
DATASETS   : climatology lra5 oras5 era5


## Check the setup
Scripts present, `gcloud` can see your bucket, and free space. (`df` on the Hub home shows the whole shared volume, not your personal quota.)

In [ ]:
%%bash
ls -la "$SCRIPTS"
gcloud storage ls "${BUCKET%/*}/" > /dev/null && echo "bucket folder reachable: ${BUCKET%/*}/" || echo "CANNOT reach ${BUCKET%/*}/ - check your path and permissions"
echo "--- free space"
df -h ~ /tmp | cut -c1-110
echo "--- home usage"
du -sh --apparent-size ~

## Start the transfer (background)
Runs the datasets one after another and **stops at the first failure**. Output goes to `notebooks/transfer.log`. The guard refuses to start a second copy. It is safe to re-run after an interruption (finished downloads are kept; the upload skips what is already in the bucket).

In [ ]:
%%bash
if pgrep -f fetch_dataset.sh > /dev/null; then
  echo "A transfer is already running; not starting another. Use the watch cell."
else
  nohup sh -c 'for d in $DATASETS; do bash "$SCRIPTS/fetch_dataset.sh" $d "$BUCKET" "$WORK" || { echo "FAILED $d"; break; }; done' > "$LOG" 2>&1 &
  echo "started: $DATASETS"
fi

## Watch progress
Re-run as often as you like. Stage markers `[1/4]`..`[4/4]`, a download line about every 30 s with its speed, and `DONE <dataset>` when a dataset is finished. `FAILED <dataset>` means it stopped: read the lines above it.

In [ ]:
%%bash
pgrep -af "fetch_dataset|download_parts|tar -xzf|storage rsync" | cut -c1-120 || echo "(not running)"
echo "--- stages so far"
grep -E "^\[[0-9]/4\]|DONE|FAILED" "$LOG" | tail -n 8
echo "--- last log lines"
tail -n 5 "$LOG" | cut -c1-160

## Verify the bucket
Checks the folder counts and compares the object count of sample folders per dataset (can take a few minutes; no output while running is normal). Expected: 16,436 day folders for the daily datasets and 8,077 objects for climatology. A `CHECK` line means look closer, not necessarily that data is missing.

In [3]:
%%bash
bash "$SCRIPTS/verify_bucket.sh" "$BUCKET"

== climatology
   PASS  objects: 8077
== era5
   PASS  day folders: 16436
   sample 1: 62 lines  (era5_full_1.5deg_19790101.zarr)
   sample 8218: 62 lines  (era5_full_1.5deg_20010701.zarr)
   sample 16436: 62 lines  (era5_full_1.5deg_20231231.zarr)
   PASS  sample folders have equal counts
== lra5
   PASS  day folders: 16436
   sample 1: 234 lines  (lra5_full_1.5deg_19790101.zarr)
   sample 8218: 234 lines  (lra5_full_1.5deg_20010701.zarr)
   sample 16436: 234 lines  (lra5_full_1.5deg_20231231.zarr)
   PASS  sample folders have equal counts
== oras5
   PASS  day folders: 16436
   sample 1: 119 lines  (oras5_full_1.5deg_19790101.zarr)
   sample 8218: 119 lines  (oras5_full_1.5deg_20010701.zarr)
   sample 16436: 119 lines  (oras5_full_1.5deg_20231231.zarr)
   PASS  sample folders have equal counts

All checks passed.


## Stop a running transfer (only if you need to)
Uncomment and run. This stops the download/extract/upload processes; starting again resumes.

In [ ]:
%%bash
# pkill -f "for d in"; pkill -f fetch_dataset.sh; pkill -f download_parts.py; pkill -f "tar -xzf"; pkill -f "storage rsync"
# pgrep -af "fetch_dataset|download_parts|tar -xzf|storage rsync" || echo "nothing running"

## Optional: open one day
Needs `xarray`, `zarr` and `gcsfs` in your environment. Replace the name with a folder that exists in your bucket.

In [ ]:
import xarray as xr
ds = xr.open_zarr(os.environ["BUCKET"] + "/era5/era5_full_1.5deg_19791208.zarr")
ds